<a href="https://colab.research.google.com/github/mbaker21231/MicroII-Sandbox/blob/main/Hopenhayn2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Code to simulate Hopenhayn model

A first requirement is a utility function to make a continuous distribution into a grid. Here it is:

In [1]:
#Packages

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

I think it is a good idea to define all the parameters we are using in one place, and basically keep track of them on the basis of whether they are globals or locals, and experimental values.

In [26]:
N   =   100
MU  =    -.25
RHO =     .85
SIGMA =   .40

# Global parameters

phi_c = 1
beta = .05

# model parameters

tolerance = 1e-10

In [2]:
def tauchen(N, mu, rho, sigma, n_std=4):
    z = np.linspace(mu - n_std * sigma / np.sqrt(1 - rho**2),
                     mu + n_std * sigma / np.sqrt(1 - rho**2), N)
    step = (z[1] - z[0])
    P = np.zeros((N, N))

    for j in range(N):
        for k in range(N):
            if k == 0:
                P[j, k] = norm.cdf((z[k] - rho * z[j] + step / 2) / sigma)
            elif k == N-1:
                P[j, k] = 1 - norm.cdf((z[k] - rho * z[j] - step / 2) / sigma)
            else:
                P[j, k] = (norm.cdf((z[k] - rho * z[j] + step / 2) / sigma) -
                           norm.cdf((z[k] - rho * z[j] - step / 2) / sigma))

    return z, P


Note that we can also use this distribution to recover a cumulative unconditional distribution, which is useful for initial productivity draws:

In [3]:
def init_dist(N, mu, rho, sigma, n_std=4):

    tauch = tauchen(N, mu, rho, sigma, n_std)
    probs = np.sum(tauch[1], axis=0)/np.sum(tauch[1])
    vals = tauch[0]

    return vals, probs

## Aspects of the model

Each firm has a flow profit function of the form:
$$
\pi(z) = \tilde z n^\alpha - Wn
$$

where $\tilde z$ is the (exponentiated) skill level $z$, $\tilde z=e^z$. Flow profits are acheived by choosing $n$, labor, to maximize th above:

$$
 \alpha \tilde z n^{\alpha -1}-W \quad \rightarrow\quad n^*(z,w) = \left(\frac{\alpha \tilde z}{W}\right)^\frac{1}{1-\alpha}
$$

Here is a function that returns, for a given wage and skill level, profits and labor demand:

In [7]:
def prof_lab(z, W):

  z_tilde = np.exp(z)
  n_sta   = ( alpha * z_tilde / W)**(1/(1-alpha))
  profs   = z_tilde*n_sta**alpha - W*alpha

  return profs, n_sta

## Present value of a firm

The following bit of code essentially iterates the value function, taking into account that the firm's valuation changes as a result of possible changes in $z$, the skill level of the firm.

In [23]:
def val_fun(z, p, W, max_iter=3000, tol=1e-10, noisy=False):

  v = np.zeros((len(z), 1))

  for i in range(max_iter):

    profs = prof_lab(z, W)[0]
    profs = np.reshape(profs, (len(z), 1))
    vnew = np.maximum( 0, profs - phi_c + (1-beta)* p @ v)
    if np.max(abs(vnew-v)<tol):
      break

  if noisy:
    print("Iterations: ", i)
    print("Maximum value: ", np.max(vnew))
    print("Average value: ", np.mean(vnew))
    print("Minimum value: ", np.min(vnew))

  return vnew

## Computing an equilibrium wage

Let's first take a stab at computing an equilibrium wage. Intuitively, we want the wage to be such that the supply of labor is equal to the total demand for labor. Where firms that do not produce exit the market.

In [44]:
Z, P = tauchen(N, MU, RHO, SIGMA)       # Skills and firms
W = .74                                   # Initial wage guess
M = 2                                   # Mass of firms
V = val_fun(Z, P, W, noisy=True)        # Value functions at skills
beven = np.argmax(V > tolerance)        # Index of first nonzero value
pi, n = prof_lab(Z[beven], W)           # Profits, labor of active firms

np.sum(n), beven


Iterations:  0
Maximum value:  176.76988011827711
Average value:  14.763083538379727
Minimum value:  0.0


(1.004479713642167, 60)

So, we are beginning to get the idea - we can increase the wage and get a resulting number of active firms. Now, to get dynamics, we need to do get some initial distribution of firms. We have:

In [45]:
Z_init, P_init = init_dist(N, MU, RHO, SIGMA)

Now, we can use this initial distribution to compute $V$:

In [49]:
P_init

array([0.00534666, 0.00182732, 0.00229122, 0.00282041, 0.0034101 ,
       0.00405196, 0.00473438, 0.00544311, 0.00616207, 0.0068745 ,
       0.00756409, 0.00821607, 0.00881819, 0.00936138, 0.00984003,
       0.01025202, 0.01059841, 0.01088288, 0.0111111 , 0.01128993,
       0.01142681, 0.01152915, 0.01160389, 0.01165721, 0.01169436,
       0.01171965, 0.01173646, 0.01174738, 0.01175431, 0.0117586 ,
       0.0117612 , 0.01176274, 0.01176362, 0.01176412, 0.0117644 ,
       0.01176455, 0.01176463, 0.01176467, 0.01176469, 0.0117647 ,
       0.0117647 , 0.0117647 , 0.01176471, 0.01176471, 0.01176471,
       0.01176471, 0.01176471, 0.01176471, 0.01176471, 0.01176471,
       0.01176471, 0.01176471, 0.01176471, 0.01176471, 0.01176471,
       0.01176471, 0.01176471, 0.01176471, 0.01176471, 0.0117647 ,
       0.0117647 , 0.0117647 , 0.01176469, 0.01176467, 0.01176464,
       0.01176457, 0.01176444, 0.0117642 , 0.01176376, 0.01176298,
       0.01176162, 0.0117593 , 0.01175545, 0.01174921, 0.01173

In [50]:
P

array([[1.23832406e-01, 3.41847223e-02, 3.98563576e-02, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [9.91466321e-02, 2.94541812e-02, 3.50333855e-02, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [7.82679673e-02, 2.49512717e-02, 3.02759269e-02, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [1.20865776e-43, 8.82854662e-43, 7.13874301e-42, ...,
        3.71470139e-02, 3.15054760e-02, 1.09476165e-01],
       [1.96289095e-44, 1.46654113e-43, 1.20976092e-42, ...,
        4.19480437e-02, 3.62947199e-02, 1.35899444e-01],
       [3.13431250e-45, 2.39512055e-44, 2.01560535e-43, ...,
        4.65725867e-02, 4.11085059e-02, 1.66387386e-01]])